In [1]:
%pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ===== SECTION 1: Imports & Setup =====
import os
import re
import math
import random
import json
from collections import Counter

import numpy as np
import pandas as pd
import kagglehub
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import gensim.downloader as gensim_api

random.seed(42)
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/opt/anaconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [3]:
# ===== SECTION 2: Load Data =====
path = kagglehub.dataset_download("eoinamoore/historical-nba-data-and-player-box-scores")

games = pd.read_csv(os.path.join(path, "Games.csv"))
team_stats = pd.read_csv(os.path.join(path, "TeamStatistics.csv"))
player_stats = pd.read_csv(os.path.join(path, "PlayerStatisticsExtended.csv"))

games["gameDate"] = pd.to_datetime(games["gameDate"])
team_stats["gameDate"] = pd.to_datetime(team_stats["gameDate"])
player_stats["gameDate"] = pd.to_datetime(player_stats["gameDateTimeEst"])

print(f"Games: {games.shape}, TeamStatistics: {team_stats.shape}, PlayerStatisticsExtended: {player_stats.shape}")

/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/3646805406.py:4: DtypeWarning: Columns (0: gameSubtype, 1: gameSubLabel, 2: seriesGameNumber, 3: arenaName, 4: arenaCity, 5: arenaState, 6: officials) have mixed types. Specify dtype option on import or set low_memory=False.
  games = pd.read_csv(os.path.join(path, "Games.csv"))
/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/3646805406.py:6: DtypeWarning: Columns (0: gameLabel, 1: gameSubLabel, 2: seriesGameNumber, 3: comment, 4: startingPosition, 5: numMinutes) have mixed types. Specify dtype option on import or set low_memory=False.
  player_stats = pd.read_csv(os.path.join(path, "PlayerStatisticsExtended.csv"))


Games: (73279, 23), TeamStatistics: (146560, 59), PlayerStatisticsExtended: (838803, 111)


In [4]:
# ===== SECTION 3: Fix corrupted teamId values =====
games_lookup = games.set_index("gameId")[["hometeamId", "awayteamId"]]

def recover_team_id(row):
    if row["teamId"] != 0:
        return row["teamId"]
    try:
        g = games_lookup.loc[row["gameId"]]
        return g["hometeamId"] if row["home"] == 1 else g["awayteamId"]
    except KeyError:
        return 0

team_stats["teamId"] = team_stats.apply(recover_team_id, axis=1)

In [5]:
# ===== SECTION 4: Leak-proof TEAM-level rolling features =====
team_stats = team_stats.sort_values(["teamId", "gameDate"]).reset_index(drop=True)

team_stats["prev5_win_pct"] = (
    team_stats.groupby("teamId")["win"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .reset_index(drop=True)
)
team_stats["prev5_avg_points"] = (
    team_stats.groupby("teamId")["teamScore"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .reset_index(drop=True)
)
team_stats["prev10_win_pct"] = (
    team_stats.groupby("teamId")["win"].apply(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
    .reset_index(drop=True)
)
team_stats["prev10_avg_points"] = (
    team_stats.groupby("teamId")["teamScore"].apply(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
    .reset_index(drop=True)
)
team_stats["days_since_last_game"] = team_stats.groupby("teamId")["gameDate"].diff().dt.days

In [6]:
# ===== SECTION 5: Leak-proof PLAYER-level rolling features =====
player_stats = player_stats.sort_values(["personId", "gameDate"]).reset_index(drop=True)

player_stats["prev5_avg_points"] = (
    player_stats.groupby("personId")["points"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .reset_index(drop=True)
)
player_stats["prev5_avg_pie"] = (
    player_stats.groupby("personId")["playerImpactEstimate"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .reset_index(drop=True)
)
player_stats["prev5_avg_assists"] = (
    player_stats.groupby("personId")["assists"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .reset_index(drop=True)
)
player_stats["prev5_avg_rebounds"] = (
    player_stats.groupby("personId")["reboundsTotal"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .reset_index(drop=True)
)

/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/3160234985.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_stats["prev5_avg_pie"] = (
/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/3160234985.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_stats["prev5_avg_assists"] = (
/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/3160234985.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

In [7]:
# ===== SECTION 6: Identify each team's featured scorer per historical game =====
def extract_specialist(df, metric_col, prefix):
    valid = df.dropna(subset=[metric_col])
    idx = valid.groupby(["gameId", "playerteamId"])[metric_col].idxmax()
    out = df.loc[idx, ["gameId", "playerteamId", "personId", "firstName", "lastName", metric_col]]
    return out.rename(columns={
        "personId": f"{prefix}_id",
        "firstName": f"{prefix}_firstName",
        "lastName": f"{prefix}_lastName",
        metric_col: f"{prefix}_value",
    })

team_top_scorer = extract_specialist(player_stats, "prev5_avg_points", "top_scorer")
team_top_playmaker = extract_specialist(player_stats, "prev5_avg_assists", "top_playmaker")
team_top_rebounder = extract_specialist(player_stats, "prev5_avg_rebounds", "top_rebounder")

In [8]:
# ===== SECTION 7: Merge into final game-level table (home/away side-by-side) =====
for specialist_df in [team_top_scorer, team_top_playmaker, team_top_rebounder]:
    specialist_df_renamed = specialist_df.rename(columns={"playerteamId": "teamId"})
    team_stats = team_stats.merge(specialist_df_renamed, on=["gameId", "teamId"], how="left")

home_rows = team_stats[team_stats["home"] == 1].add_prefix("home_")
away_rows = team_stats[team_stats["home"] == 0].add_prefix("away_")

game_level = home_rows.merge(away_rows, left_on="home_gameId", right_on="away_gameId", how="inner")
game_level["label_home_win"] = game_level["home_win"]

print(f"Game-level table (before filtering): {game_level.shape}")

Game-level table (before filtering): (73281, 153)


/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/2842013424.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  game_level["label_home_win"] = game_level["home_win"]


In [9]:
# ===== SECTION 8: Filter to complete cases =====
numeric_cols = [
    "home_prev5_win_pct", "home_prev5_avg_points",
    "home_prev10_win_pct", "home_prev10_avg_points",
    "home_top_scorer_value", "home_top_playmaker_value", "home_top_rebounder_value",
    "away_prev5_win_pct", "away_prev5_avg_points",
    "away_prev10_win_pct", "away_prev10_avg_points",
    "away_top_scorer_value", "away_top_playmaker_value", "away_top_rebounder_value",
]

final_df = game_level.dropna(subset=numeric_cols + ["label_home_win"]).reset_index(drop=True)
final_df["home_gameDate"] = pd.to_datetime(final_df["home_gameDate"])

print(f"Final clean training table: {final_df.shape}")
print(final_df["home_gameDate"].dt.year.value_counts().sort_index())

Final clean training table: (36085, 153)
home_gameDate
1996     407
1997    1262
1998     837
1999    1213
2000     842
2001    1095
2002    1275
2003    1279
2004    1247
2005    1322
2006    1341
2007    1309
2008    1332
2009    1316
2010    1321
2011     885
2012    1474
2013    1324
2014    1334
2015    1319
2016    1333
2017    1347
2018    1316
2019    1267
2020     707
2021    1104
2022     547
2023    1254
2024    1322
2025    1326
2026     828
Name: count, dtype: int64


In [10]:
# ===== SECTION 9: Generate pregame narrative text =====
def generate_training_narrative(row):
    home_team = f"{row['home_teamCity']} {row['home_teamName']}"
    away_team = f"{row['away_teamCity']} {row['away_teamName']}"
    home_win_pct = row["home_prev5_win_pct"] * 100
    away_win_pct = row["away_prev5_win_pct"] * 100
    home_scorer = f"{row['home_top_scorer_firstName']} {row['home_top_scorer_lastName']}"
    away_scorer = f"{row['away_top_scorer_firstName']} {row['away_top_scorer_lastName']}"
    home_playmaker = f"{row['home_top_playmaker_firstName']} {row['home_top_playmaker_lastName']}"
    away_playmaker = f"{row['away_top_playmaker_firstName']} {row['away_top_playmaker_lastName']}"
    home_rebounder = f"{row['home_top_rebounder_firstName']} {row['home_top_rebounder_lastName']}"
    away_rebounder = f"{row['away_top_rebounder_firstName']} {row['away_top_rebounder_lastName']}"
    favorite = home_team if home_win_pct >= away_win_pct else away_team

    templates = [
        (
            f"Tonight's matchup pits {home_team} against {away_team}, and the numbers favor {favorite} heading in. "
            f"{home_team} have won {home_win_pct:.0f} percent of their last five games, led by {home_scorer} scoring "
            f"and {home_playmaker} setting the table. {away_team} have gone {away_win_pct:.0f} percent over their last five, "
            f"relying on {away_scorer} for buckets and {away_rebounder} on the glass."
        ),
        (
            f"{home_team} host {away_team} in a matchup with plenty of storylines. {home_team} enter at {home_win_pct:.0f} percent "
            f"over their last five, with {home_rebounder} controlling the boards and {home_scorer} leading the scoring column. "
            f"{away_team} counter at {away_win_pct:.0f} percent, powered by {away_playmaker} distributing and {away_scorer} finishing."
        ),
        (
            f"All eyes are on {home_team} versus {away_team} tonight. {home_team} bring a {home_win_pct:.0f} percent mark over their "
            f"last five games, anchored by {home_scorer} and {home_playmaker}. {away_team} arrive at {away_win_pct:.0f} percent, "
            f"with {away_rebounder} dominating inside and {away_scorer} carrying the scoring load."
        ),
        (
            f"{home_team} and {away_team} square off in what should be a competitive night. Over their last five games, "
            f"{home_team} sit at {home_win_pct:.0f} percent behind {home_scorer}, while {away_team} check in at {away_win_pct:.0f} percent "
            f"thanks to strong two-way play from {away_playmaker} and {away_rebounder}."
        ),
    ]
    return random.choice(templates)

final_df["pregame_text"] = final_df.apply(generate_training_narrative, axis=1)
print(final_df[["pregame_text"]].sample(2, random_state=1))


                                            pregame_text
33455  All eyes are on Washington Wizards versus Toro...
29851  Tonight's matchup pits Toronto Raptors against...


/var/folders/v0/kd_912h901d7d8n82m_pvbjr0000gn/T/ipykernel_2803/1502428095.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df["pregame_text"] = final_df.apply(generate_training_narrative, axis=1)


In [11]:
# =============================================================
# SECTION 10 (REPLACE ENTIRELY): Smarter, graded, combinable
# synthetic text injection tied to real pregame-gap "surprise" intensity.
# =============================================================
def inject_upset_signal(row):
    home_favored = row["home_prev5_win_pct"] > row["away_prev5_win_pct"]
    home_won = row["label_home_win"] == 1.0
    upset = (home_favored and not home_won) or (not home_favored and home_won)

    # How big was the pregame gap between the two teams? (0 = even matchup, up to ~1 = huge mismatch)
    pregame_gap = abs(row["home_prev5_win_pct"] - row["away_prev5_win_pct"])
    surprise = pregame_gap if upset else 0.0

    severe_phrases = [
        " The team's leading scorer will not play tonight due to a season-ending injury.",
        " Multiple starters are ruled out, forcing the team to rely on bench depth.",
        " The head coach confirmed the star player will sit out indefinitely.",
        " A significant trade this week has completely reshuffled the starting lineup.",
        " The team is on the second night of a back-to-back after a grueling road trip.",
        " Reports confirm the All-Star forward suffered a fracture and is out for months.",
        " The franchise announced their best player will miss the rest of the season.",
        " A locker room dispute reportedly led to two key players being benched.",
        " The starting center was ejected from the previous game and remains suspended.",
        " Three rotation players are simultaneously dealing with separate injuries.",
        " The team fired its head coach just days before this matchup.",
        " A key defender is out due to a positive illness test.",
        " The star point guard reportedly requested a trade and skipped shootaround.",
        " Weather delays forced the team to travel through the night before the game.",
        " The team's best perimeter defender tore a ligament in the previous game.",
    ]
    moderate_phrases = [
        " Reports indicate a key rotation player is questionable with a lower-body injury.",
        " The team has looked fatigued after a difficult stretch of games.",
        " A starting guard is listed as doubtful with a hamstring strain.",
        " The team is playing its fourth game in five nights.",
        " There are whispers of tension between the coaching staff and a veteran player.",
        " A newly acquired player is still adjusting to the system after a recent trade.",
        " The team's second-leading scorer is questionable with knee soreness.",
        " A key reserve has missed the last two games with a minor ailment.",
        " The team has struggled on the road this season, going 3-8 away from home.",
        " A rotation player is in the concussion protocol after the last game.",
        " The bench has been inconsistent, scoring in single digits in three of the last five games.",
        " The team's starting lineup has changed twice in the past week.",
        " A veteran leader is dealing with soreness but is expected to play limited minutes.",
        " The team is adjusting to a new starting lineup after a recent trade.",
        " There are minor concerns about spacing after a recent rotation change.",
    ]
    mild_phrases = [
        " There are locker room concerns about team chemistry heading into this matchup.",
        " A role player was listed as probable after a minor tweak in practice.",
        " The team is coming off a short rest but reports no injury concerns.",
        " Analysts note a minor lineup change is possible but unlikely to matter much.",
        " A bench player missed shootaround for personal reasons unrelated to injury.",
        " The team traveled a long distance but downplayed any fatigue concerns.",
        " A reserve big man is probable after resting in the previous game.",
        " Coaches say the rotation could shorten slightly for this matchup.",
        " The team practiced lightly the day before the game as a precaution.",
        " There is minor speculation about a lineup tweak, but nothing is confirmed.",
        " A role player is probable after being a late addition to the injury report.",
        " The team downplayed reports of minor soreness among two reserves.",
        " Analysts see this as a low-stakes measuring-stick game for both sides.",
        " The matchup has generated modest buzz among local reporters.",
        " Both teams enter this game with relatively clean injury reports.",
    ]
    positive_phrases = [
        " The team enters this game with strong momentum after several improved performances.",
        " A key player returned from injury and looked sharp in the last outing.",
        " The starting lineup has stayed fully healthy and consistent in recent games.",
        " Coaches praised the team's improved defensive intensity in practice this week.",
        " The team is well-rested after several days off before this matchup.",
        " A recent roster addition has boosted depth and morale heading into the game.",
        " The team is riding a season-best winning streak entering tonight.",
        " Analysts highlight improved ball movement and shooting efficiency lately.",
        " The star player was named player of the week after a dominant stretch.",
        " The team's bench production has been a bright spot in recent wins.",
        " Confidence is high after a statement win over a top playoff contender.",
        " The coaching staff credits a tactical adjustment for the recent turnaround.",
        " The team's home record has been excellent this season.",
        " A returning veteran has stabilized the second unit significantly.",
        " Team chemistry looks strong following a players-only meeting last week.",
    ]

    def combo(phrase_list):
        k = random.choice([1, 1, 2])  # usually one phrase, sometimes two combined
        return " ".join(random.sample(phrase_list, k=k))

    text = row["pregame_text"]
    r = random.random()

    if upset:
        if surprise > 0.35 and r < 0.55:
            text += " " + combo(severe_phrases)
        elif surprise > 0.15 and r < 0.55:
            text += " " + combo(moderate_phrases)
        elif r < 0.45:
            text += " " + combo(mild_phrases)
        # otherwise: upset with no textual cue at all - keeps model honest, still relies on stats
    else:
        if r < 0.06:
            text += " " + combo(mild_phrases)
        elif r < 0.12:
            text += " " + combo(positive_phrases)

    return text

final_df_ablation = final_df.copy()
final_df_ablation["pregame_text"] = final_df_ablation.apply(inject_upset_signal, axis=1)
print("Smarter ablation dataset created (graded severity + combinable phrases).")
print(final_df_ablation[["pregame_text"]].sample(3, random_state=1))

Smarter ablation dataset created (graded severity + combinable phrases).
                                            pregame_text
33455  All eyes are on Washington Wizards versus Toro...
29851  Tonight's matchup pits Toronto Raptors against...
12830  Los Angeles Lakers host Utah Jazz in a matchup...


In [12]:
# ===== SECTION 11: Tokenizer / vocabulary (main, realistic-text model) =====
def simple_tokenize(text):
    return re.findall(r"[a-z0-9]+|[.,%]", text.lower())

all_tokens = []
for text in final_df["pregame_text"]:
    all_tokens.extend(simple_tokenize(text))
token_counts = Counter(all_tokens)

vocab = {"<PAD>": 0, "<UNK>": 1}
for token, count in token_counts.most_common():
    vocab[token] = len(vocab)

print(f"Vocab size: {len(vocab)}")

def encode_text(text, vocab, max_len=100):
    tokens = simple_tokenize(text)
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens][:max_len]
    attention_mask = [1] * len(ids)
    pad_len = max_len - len(ids)
    ids = ids + [vocab["<PAD>"]] * pad_len
    attention_mask = attention_mask + [0] * pad_len
    return ids, attention_mask

Vocab size: 1978


In [13]:
# ===== SECTION 12: Time-based train/val split + standardization (main model) =====
train_df = final_df[final_df["home_gameDate"] < "2024-01-01"].reset_index(drop=True)
val_df = final_df[final_df["home_gameDate"] >= "2024-01-01"].reset_index(drop=True)

train_means = train_df[numeric_cols].mean()
train_stds = train_df[numeric_cols].std()

def standardize(df):
    return (df[numeric_cols] - train_means) / train_stds

train_numeric = standardize(train_df).values.astype("float32")
val_numeric = standardize(val_df).values.astype("float32")

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

Train size: 32609, Val size: 3476


In [14]:
# ===== SECTION 13: PyTorch Dataset & DataLoader (main model) =====
class NBADataset(Dataset):
    def __init__(self, texts, numeric_features, labels, vocab, max_len=100):
        self.texts = texts
        self.numeric_features = numeric_features
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, mask = encode_text(self.texts[idx], self.vocab, self.max_len)
        return {
            "token_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
            "numeric_features": torch.tensor(self.numeric_features[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

train_dataset = NBADataset(train_df["pregame_text"].tolist(), train_numeric, train_df["label_home_win"].values, vocab, max_len=100)
val_dataset = NBADataset(val_df["pregame_text"].tolist(), val_numeric, val_df["label_home_win"].values, vocab, max_len=100)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
# ===== SECTION 14: Model architecture (GloVe-ready) =====
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


def build_embedding_matrix(vocab, embedding_dim=100):
    print("Downloading/loading pretrained GloVe vectors (one-time, cached after first run)...")
    glove = gensim_api.load(f"glove-wiki-gigaword-{embedding_dim}")

    matrix = np.random.normal(scale=0.6, size=(len(vocab), embedding_dim)).astype("float32")
    matrix[0] = np.zeros(embedding_dim)  # <PAD> stays zero

    found = 0
    for word, idx in vocab.items():
        w = word.lower()
        if w in glove:
            matrix[idx] = glove[w]
            found += 1
    print(f"Found pretrained vectors for {found}/{len(vocab)} words ({found / len(vocab) * 100:.1f}%)")
    return torch.tensor(matrix, dtype=torch.float32)


class TextEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=1,
                 dim_feedforward=256, dropout=0.1, max_len=256, pretrained_embeddings=None):
        super().__init__()
        self.d_model = d_model
        if pretrained_embeddings is not None:
            self.token_embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=False, padding_idx=0
            )
        else:
            self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, token_ids, attention_mask=None):
        x = self.token_embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        encoded = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            text_repr = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            text_repr = encoded.mean(dim=1)
        return text_repr

    def get_attention_weights(self, token_ids, attention_mask=None):
        x = self.token_embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        self_attn = self.transformer.layers[0].self_attn
        key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        _, attn_weights = self_attn(x, x, x, key_padding_mask=key_padding_mask,
                                     need_weights=True, average_attn_weights=True)
        return attn_weights


class StatsEncoder(nn.Module):
    def __init__(self, n_numeric_features, hidden_dim=32, out_dim=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_numeric_features, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )

    def forward(self, numeric_features):
        return self.net(numeric_features)


class DualBranchFusionModel(nn.Module):
    def __init__(self, vocab_size, n_numeric_features, text_dim=128, stats_dim=32,
                 n_heads=4, n_text_layers=1, classifier_hidden=64, dropout=0.1, max_len=256,
                 pretrained_embeddings=None):
        super().__init__()
        self.text_encoder = TextEncoder(vocab_size, text_dim, n_heads, n_text_layers, dropout=dropout,
                                         max_len=max_len, pretrained_embeddings=pretrained_embeddings)
        self.stats_encoder = StatsEncoder(n_numeric_features, out_dim=stats_dim, dropout=dropout)
        fusion_dim = text_dim + stats_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, classifier_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(classifier_hidden, 1),
        )

    def forward(self, token_ids, numeric_features, attention_mask=None):
        text_repr = self.text_encoder(token_ids, attention_mask)
        stats_repr = self.stats_encoder(numeric_features)
        fused = torch.cat([text_repr, stats_repr], dim=1)
        return self.classifier(fused).squeeze(-1)

In [16]:
# ===== SECTION 15: Train the MAIN model (realistic, auto-generated text) =====
model = DualBranchFusionModel(vocab_size=len(vocab), n_numeric_features=len(numeric_cols)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

def evaluate(loader, m):
    m.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            token_ids = batch["token_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            numeric_features = batch["numeric_features"].to(device)
            labels = batch["label"].to(device)
            logits = m(token_ids, numeric_features, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        token_ids = batch["token_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric_features"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(token_ids, numeric_features, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
    train_loss /= len(train_dataset)
    val_loss, val_acc = evaluate(val_loader, model)
    print(f"[MAIN] Epoch {epoch+1}/{n_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

/opt/anaconda3/envs/py311/lib/python3.11/site-packages/torch/nn/modules/transformer.py:384: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:179.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


[MAIN] Epoch 1/5 | train_loss=0.6392 | val_loss=0.6378 | val_acc=0.6335
[MAIN] Epoch 2/5 | train_loss=0.6325 | val_loss=0.6373 | val_acc=0.6303
[MAIN] Epoch 3/5 | train_loss=0.6318 | val_loss=0.6387 | val_acc=0.6280
[MAIN] Epoch 4/5 | train_loss=0.6305 | val_loss=0.6360 | val_acc=0.6326
[MAIN] Epoch 5/5 | train_loss=0.6303 | val_loss=0.6381 | val_acc=0.6237


In [17]:
# ===== SECTION 16: Baseline comparison (main model, numeric only) =====
baseline = LogisticRegression(max_iter=1000)
baseline.fit(train_numeric, train_df["label_home_win"].values)
baseline_preds = baseline.predict(val_numeric)
baseline_acc = accuracy_score(val_df["label_home_win"].values, baseline_preds)
print(f"\nBaseline (numeric only) val accuracy: {baseline_acc:.4f}")


Baseline (numeric only) val accuracy: 0.6326


In [19]:
# ===== SECTION 16.5: Ablation pipeline (graded signal + GloVe embeddings) =====
all_tokens_ablation = []
for text in final_df_ablation["pregame_text"]:
    all_tokens_ablation.extend(simple_tokenize(text))
token_counts_ablation = Counter(all_tokens_ablation)

vocab_ablation = {"<PAD>": 0, "<UNK>": 1}
for token, count in token_counts_ablation.most_common():
    vocab_ablation[token] = len(vocab_ablation)

print(f"Ablation vocab size: {len(vocab_ablation)}")

train_df_abl = final_df_ablation[final_df_ablation["home_gameDate"] < "2024-01-01"].reset_index(drop=True)
val_df_abl = final_df_ablation[final_df_ablation["home_gameDate"] >= "2024-01-01"].reset_index(drop=True)

train_means_abl = train_df_abl[numeric_cols].mean()
train_stds_abl = train_df_abl[numeric_cols].std()

def standardize_abl(df):
    return (df[numeric_cols] - train_means_abl) / train_stds_abl

train_numeric_abl = standardize_abl(train_df_abl).values.astype("float32")
val_numeric_abl = standardize_abl(val_df_abl).values.astype("float32")

train_dataset_abl = NBADataset(train_df_abl["pregame_text"].tolist(), train_numeric_abl,
                                train_df_abl["label_home_win"].values, vocab_ablation, max_len=100)
val_dataset_abl = NBADataset(val_df_abl["pregame_text"].tolist(), val_numeric_abl,
                              val_df_abl["label_home_win"].values, vocab_ablation, max_len=100)
train_loader_abl = DataLoader(train_dataset_abl, batch_size=32, shuffle=True)
val_loader_abl = DataLoader(val_dataset_abl, batch_size=32, shuffle=False)

embedding_matrix_ablation = build_embedding_matrix(vocab_ablation, embedding_dim=100)

model_ablation = DualBranchFusionModel(
    vocab_size=len(vocab_ablation),
    n_numeric_features=len(numeric_cols),
    text_dim=100,
    pretrained_embeddings=embedding_matrix_ablation,
).to(device)

optimizer_abl = torch.optim.Adam(model_ablation.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_abl = nn.BCEWithLogitsLoss()

def evaluate_abl(loader, m):
    m.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            token_ids = batch["token_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            numeric_features = batch["numeric_features"].to(device)
            labels = batch["label"].to(device)
            logits = m(token_ids, numeric_features, attention_mask)
            loss = criterion_abl(logits, labels)
            total_loss += loss.item() * len(labels)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

import copy

n_epochs_abl = 10
best_val_acc = 0.0
best_state_dict = None

for epoch in range(n_epochs_abl):
    model_ablation.train()
    train_loss = 0
    for batch in train_loader_abl:
        token_ids = batch["token_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric_features"].to(device)
        labels = batch["label"].to(device)
        optimizer_abl.zero_grad()
        logits = model_ablation(token_ids, numeric_features, attention_mask)
        loss = criterion_abl(logits, labels)
        loss.backward()
        optimizer_abl.step()
        train_loss += loss.item() * len(labels)
    train_loss /= len(train_dataset_abl)
    val_loss_abl, val_acc_abl = evaluate_abl(val_loader_abl, model_ablation)
    print(f"[ABLATION+GloVe] Epoch {epoch+1}/{n_epochs_abl} | train_loss={train_loss:.4f} | val_loss={val_loss_abl:.4f} | val_acc={val_acc_abl:.4f}")

    if val_acc_abl > best_val_acc:
        best_val_acc = val_acc_abl
        best_state_dict = copy.deepcopy(model_ablation.state_dict())

model_ablation.load_state_dict(best_state_dict)
print(f"\nRestored best checkpoint: val_acc={best_val_acc:.4f}")
baseline_abl = LogisticRegression(max_iter=1000)
baseline_abl.fit(train_numeric_abl, train_df_abl["label_home_win"].values)
baseline_abl_preds = baseline_abl.predict(val_numeric_abl)
baseline_abl_acc = accuracy_score(val_df_abl["label_home_win"].values, baseline_abl_preds)

print(f"\n=== COMPARISON ===")
print(f"Ablation model val_acc (best):      {best_val_acc:.4f}")
print(f"Numeric-only baseline (ablation):  {baseline_abl_acc:.4f}")

Ablation vocab size: 2237
Downloading/loading pretrained GloVe vectors (one-time, cached after first run)...
Found pretrained vectors for 2063/2237 words (92.2%)
[ABLATION+GloVe] Epoch 1/10 | train_loss=0.5119 | val_loss=0.4388 | val_acc=0.7848
[ABLATION+GloVe] Epoch 2/10 | train_loss=0.4264 | val_loss=0.4340 | val_acc=0.7851
[ABLATION+GloVe] Epoch 3/10 | train_loss=0.4213 | val_loss=0.4261 | val_acc=0.7871
[ABLATION+GloVe] Epoch 4/10 | train_loss=0.4150 | val_loss=0.4248 | val_acc=0.7911
[ABLATION+GloVe] Epoch 5/10 | train_loss=0.4134 | val_loss=0.4292 | val_acc=0.7920
[ABLATION+GloVe] Epoch 6/10 | train_loss=0.4135 | val_loss=0.4462 | val_acc=0.7880
[ABLATION+GloVe] Epoch 7/10 | train_loss=0.4197 | val_loss=0.4263 | val_acc=0.7906
[ABLATION+GloVe] Epoch 8/10 | train_loss=0.4158 | val_loss=0.4316 | val_acc=0.7894
[ABLATION+GloVe] Epoch 9/10 | train_loss=0.4142 | val_loss=0.4267 | val_acc=0.7917
[ABLATION+GloVe] Epoch 10/10 | train_loss=0.4073 | val_loss=0.4281 | val_acc=0.7865

Restor

In [20]:
# ===== SECTION 16.6: Temperature calibration =====
def find_temperature(m, loader):
    m.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            token_ids = batch["token_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            numeric_features = batch["numeric_features"].to(device)
            labels = batch["label"].to(device)
            logits = m(token_ids, numeric_features, attention_mask)
            all_logits.append(logits)
            all_labels.append(labels)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    crit = nn.BCEWithLogitsLoss()
    best_T, best_loss = 1.0, float("inf")
    for T in np.arange(0.5, 15.0, 0.1):
        loss = crit(all_logits / T, all_labels).item()
        if loss < best_loss:
            best_loss = loss
            best_T = T
    return best_T

temperature = find_temperature(model_ablation, val_loader_abl)
print(f"Optimal temperature: {temperature:.2f}")

Optimal temperature: 1.10


In [23]:
# ===== SECTION 17: Save all artifacts for the Streamlit app =====
torch.save(model_ablation.state_dict(), "trained_model.pt")

with open("vocab.json", "w") as f:
    json.dump(vocab_ablation, f)

standardization_stats = {
    "means": train_means_abl.to_dict(),
    "stds": train_stds_abl.to_dict(),
}
with open("standardization_stats.json", "w") as f:
    json.dump(standardization_stats, f)

with open("calibration.json", "w") as f:
    json.dump({"temperature": temperature}, f)

games_per_team = team_stats["teamId"].value_counts()
valid_team_ids = games_per_team[games_per_team >= 500].index

latest_team_stats = (
    team_stats.sort_values("gameDate")
    .groupby("teamId")
    .tail(1)[[
        "teamId", "teamCity", "teamName",
        "prev5_win_pct", "prev5_avg_points",
        "prev10_win_pct", "prev10_avg_points",
        "top_scorer_firstName", "top_scorer_lastName", "top_scorer_value",
        "top_playmaker_firstName", "top_playmaker_lastName", "top_playmaker_value",
        "top_rebounder_firstName", "top_rebounder_lastName", "top_rebounder_value",
    ]]
    .reset_index(drop=True)
)
latest_team_stats = latest_team_stats[
    latest_team_stats["teamId"].isin(valid_team_ids)
].reset_index(drop=True)

print(f"Real NBA franchises kept: {len(latest_team_stats)}")
latest_team_stats.to_csv("latest_team_stats.csv", index=False)

print("\nSaved: trained_model.pt, vocab.json, standardization_stats.json, calibration.json, latest_team_stats.csv")
print(latest_team_stats.head())

Real NBA franchises kept: 30

Saved: trained_model.pt, vocab.json, standardization_stats.json, calibration.json, latest_team_stats.csv
       teamId    teamCity   teamName  prev5_win_pct  prev5_avg_points  \
0  1610612754     Indiana     Pacers            0.2             107.4   
1  1610612751    Brooklyn       Nets            0.4             105.2   
2  1610612749   Milwaukee      Bucks            0.4             111.6   
3  1610612764  Washington    Wizards            0.0             114.8   
4  1610612763     Memphis  Grizzlies            0.0             111.4   

   prev10_win_pct  prev10_avg_points top_scorer_firstName top_scorer_lastName  \
0             0.4              118.8                  Obi              Toppin   
1             0.3              103.2                 E.J.             Liddell   
2             0.3              110.1               Cormac                Ryan   
3             0.1              115.3                 Will               Riley   
4             0.1    